In [191]:
import scipy.io as sio
from scipy.io.matlab import mat_struct

def _check_keys(dict):
    for key in dict:
        if isinstance(dict[key], mat_struct):
            dict[key] = _todict(dict[key])
    return dict

def _todict(matobj):
    dict = {}
    for strg in matobj._fieldnames:
        elem = matobj.__dict__[strg]
        if isinstance(elem, mat_struct):
            dict[strg] = _todict(elem)
        else:
            dict[strg] = elem
    return dict

def loadmat(filename):
    data = sio.loadmat(filename, struct_as_record=False, squeeze_me=True)
    return _check_keys(data)


In [192]:
import os
import scipy.io as sio
from scipy.io.matlab import mat_struct

# --- Your working loadmat helpers ---
def _check_keys(dict):
    for key in dict:
        if isinstance(dict[key], mat_struct):
            dict[key] = _todict(dict[key])
    return dict

def _todict(matobj):
    
    dict = {}
    for strg in matobj._fieldnames:
        elem = matobj.__dict__[strg]
        if isinstance(elem, mat_struct):
            dict[strg] = _todict(elem)
        else:
            dict[strg] = elem
    return dict

def loadmat(filename):
    data = sio.loadmat(filename, struct_as_record=False, squeeze_me=True)
    return _check_keys(data)

# --- Batch loader ---
import re  # pattern matching

def load_all_subjects(base_folder):
    all_data = {}
    
    for subj_name in os.listdir(base_folder):
        subj_path = os.path.join(base_folder, subj_name)
        # Keep only directories that start with 'Subject'
        if not os.path.isdir(subj_path) or not re.match(r'^Subject\d+$', subj_name):
            continue
        
        all_data[subj_name] = {}
        
        for cond_name in ['Control', 'Suit']:  # adjust if more conditions
            cond_path = os.path.join(subj_path, cond_name)
            mat_file = os.path.join(cond_path, 'data_fixed.mat')
            
            if os.path.exists(mat_file):
                all_data[subj_name][cond_name] = loadmat(mat_file)
            else:
                print(f"Warning: {mat_file} not found")
    
    return all_data


In [193]:
base_folder = r"//iowa.uiowa.edu/shared/ResearchData/rdss_rvitali/Sam_Files/Research/NREIP Fire Study"
all_data = load_all_subjects(base_folder)

In [194]:
# updating keys to be [(subject, condition, activity), ...]
all_data_flat = {}

for subj in all_data.keys():
    for cond in all_data[subj].keys():
        data = all_data[subj][cond]['data']
        for activity in data.keys():
            all_data_flat[(subj, cond, activity)] = data[activity]


In [195]:
import numpy as np
import pandas as pd
from scipy import signal, interpolate
import warnings
warnings.filterwarnings("ignore")

subject_masses = {
    "Subject1": 61.23,
    "Subject2": 78.02,
}

fs_ke = 128
fs_emg = 1000.0
fs_deriv = 1000.0
fs_hr_target = 10.0
channels = [1, 2, 11, 12]

def rms(x):
    if len(x) == 0:
        return np.nan
    return np.sign(np.mean(x)) * np.sqrt(np.mean(x**2))

def num_derivative(t, y):
    if len(y) < 3:
        return np.array([])
    return np.gradient(y, t)

b_emg_deriv, a_emg_deriv = signal.butter(4, [50/(fs_deriv/2), 150/(fs_deriv/2)], btype='band')
b_hr, a_hr = signal.butter(2, [0.01/(fs_hr_target/2), 0.5/(fs_hr_target/2)], btype='band')


all_rows = []
print("Processing all subjects and activities...")

for subj, cond, act in all_data_flat.keys():
    print(f"{subj} | {cond} | {act}")

    data = all_data_flat[(subj, cond, act)]

    emg_time = data['EMG'][:, 0]
    t_start, t_end = emg_time[0], emg_time[-1]
    
    bin_edges = np.arange(t_start, t_end, 15.0)
    if t_end - bin_edges[-1] > 10:
        bin_edges = np.append(bin_edges, bin_edges[-1] + 15.0)
    else:
        bin_edges = np.append(bin_edges, t_end)

    subject_mass = subject_masses[subj]
    mNorm = subject_mass / 82.2

    segments = [
        {'Name':'W','I':0.02654,'mass':0,'dist':0,'cols':[3,4,5],'src':'back'},
        {'Name':'C','I':0.33201,'mass':0,'dist':0,'cols':[3,4,5],'src':'neck'},
        {'Name':'AL','I':0.06397,'mass':0,'dist':0,'cols':[3,4,5],'src':'lshank'},
        {'Name':'AR','I':0.06271,'mass':0,'dist':0,'cols':[3,4,5],'src':'rshank'}
    ]

    KE_df = pd.DataFrame()
    for s in segments:
        acc = data['IMU'][s['src']]
        x,y,z = [np.nan_to_num(acc[:,c]) for c in s['cols']]
        omega = np.sqrt(x**2 + y**2 + z**2)
        KE_df[s['Name']] = 0.5 * (s['I']*mNorm) * omega**2

    KE_df['total'] = KE_df.sum(axis=1)

    metab = data['HRandTemp']
    hr_time = metab[:,0]
    hr = metab[:,4]

    t_hr = np.linspace(hr_time[0], hr_time[-1], len(hr_interp))
    hr_interp = interpolate.interp1d(hr_time, hr, fill_value="extrapolate")(t_hr)
    hr_filt = signal.filtfilt(b_hr, a_hr, hr_interp)

    emg = data['EMG']
    emg_lp, emg_der = {}, {}

    for ch in channels:
        raw = np.nan_to_num(emg[:,ch])
        b,a = signal.butter(5,[30/(fs_emg/2),350/(fs_emg/2)],btype='bandpass')
        f = signal.filtfilt(b,a,raw)
        r = np.abs(f)
        b2,a2 = signal.butter(4,5/(fs_emg/2),'low')
        emg_lp[ch] = signal.filtfilt(b2,a2,r)
        emg_der[ch] = np.nan_to_num(signal.filtfilt(b_emg_deriv,a_emg_deriv,raw))

    for i in range(len(bin_edges)-1):
        t0, t1 = bin_edges[i], bin_edges[i+1]

        imu_time = data['IMU']['Timestamps']
        ke0 = np.searchsorted(imu_time, t0)
        ke1 = np.searchsorted(imu_time, t1)

        emg0, emg1 = np.searchsorted(emg_time,t0), np.searchsorted(emg_time,t1)
        
        hr0 = np.searchsorted(t_hr, t0)
        hr1 = np.searchsorted(t_hr, t1)
        sig = hr_filt[hr0:hr1]


        row = {
            'Subject':subj, 'Condition':cond, 'Activity':act,
            'Time':t0, 'Weight':subject_mass,
            'KE':np.nanmean(KE_df['total'][ke0:ke1])
        }

        row['HR'] = np.nanmean(hr[(hr_time>=t0)&(hr_time<t1)])

        sig = hr_filt[hr0:hr1]
        d = num_derivative(np.arange(len(sig))/fs_hr_target, sig)
        row['HRDeriv'] = rms(d)

        for ch in channels:
            row[f'EMG{ch}'] = np.nanmean(emg_lp[ch][emg0:emg1])
            d = num_derivative(np.arange(emg1-emg0)/fs_deriv, emg_der[ch][emg0:emg1])
            row[f'EMG{ch}Deriv'] = rms(d)

        all_rows.append(row)

merged_df = pd.DataFrame(all_rows)

merged_df = merged_df.rename(columns={
    'EMG1':'rsol','EMG2':'lsol','EMG11':'rbf','EMG12':'lbf',
    'EMG1Deriv':'rsolDeriv','EMG2Deriv':'lsolDeriv',
    'EMG11Deriv':'rbfDeriv','EMG12Deriv':'lbfDeriv'
})

final_cols = ['Subject','Condition','Activity','Time','KE','HR','HRDeriv','Weight',
              'rsol','lsol','rbf','lbf','rsolDeriv','lsolDeriv','rbfDeriv','lbfDeriv']

merged_df = merged_df[final_cols]

print("Done:", merged_df.shape)


Processing all subjects and activities...
Subject1 | Control | Jog
Subject1 | Control | Wheel
Subject1 | Control | Fire
Subject1 | Control | Dummy
Subject1 | Control | Stairs
Subject1 | Control | Walk
Subject1 | Suit | Jog
Subject1 | Suit | Wheel
Subject1 | Suit | Fire
Subject1 | Suit | Dummy
Subject1 | Suit | Stairs
Subject1 | Suit | Walk
Subject2 | Control | Jog
Subject2 | Control | Wheel
Subject2 | Control | Fire
Subject2 | Control | Dummy
Subject2 | Control | Stairs
Subject2 | Control | Walk
Subject2 | Suit | Jog
Subject2 | Suit | Wheel
Subject2 | Suit | Fire
Subject2 | Suit | Dummy
Subject2 | Suit | Stairs
Subject2 | Suit | Walk
Done: (244, 16)


In [196]:
merged_df['KE'] = merged_df['KE'] * (np.pi/180)**2
print(merged_df.head(50))

     Subject Condition Activity    Time        KE          HR   HRDeriv  \
0   Subject1   Control      Jog   152.5  2.353964  108.533333  1.325205   
1   Subject1   Control      Jog   167.5  2.390222  118.533333 -0.561437   
2   Subject1   Control      Jog   182.5  2.342443  132.733333  0.752837   
3   Subject1   Control      Jog   197.5  2.519694  138.866667 -0.387906   
4   Subject1   Control      Jog   212.5  2.513439  141.400000 -0.328972   
5   Subject1   Control      Jog   227.5  2.575013  147.800000  0.443468   
6   Subject1   Control      Jog   242.5  2.619508  152.333333 -0.318061   
7   Subject1   Control      Jog   257.5  2.553851  152.466667  0.323176   
8   Subject1   Control      Jog   272.5  2.529713  152.733333 -0.251876   
9   Subject1   Control      Jog   287.5  2.542477  149.066667 -0.310395   
10  Subject1   Control      Jog   302.5  2.531956  148.800000  0.363586   
11  Subject1   Control      Jog   317.5  2.630169  148.000000 -0.154662   
12  Subject1   Control   

In [197]:
# Save DataFrame to pickle file
merged_df.to_pickle("/Users/katbutler/MetabolicCost/merged_test_df.pkl")